In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "scikit-learn", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics.pairwise import paired_cosine_distances

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
bucket_column = "max_len_bucket"
selected_bucket = "13_20"
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
    "bucket_column": bucket_column,
    "selected_bucket": selected_bucket,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()

df["sentence1_len"] = df["sentence1"].astype(str).str.split().str.len()
df["sentence2_len"] = df["sentence2"].astype(str).str.split().str.len()
df["max_pair_len"] = df[["sentence1_len", "sentence2_len"]].max(axis=1)
df[bucket_column] = pd.cut(
    df["max_pair_len"],
    bins=[0, 8, 12, 20, 10**9],
    labels=["1_8", "9_12", "13_20", "21_plus"],
    include_lowest=True,
).astype(str)

bucket_counts = df[bucket_column].value_counts().sort_index()
bucket_df = df[df[bucket_column] == selected_bucket].reset_index(drop=True)

print({"num_examples_full_validation": len(df), "num_examples_selected_bucket": len(bucket_df)})
print(bucket_counts.to_dict())
print(bucket_df.head())


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


In [ ]:
sentences1 = bucket_df["sentence1"].tolist()
sentences2 = bucket_df["sentence2"].tolist()
labels = bucket_df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

embedding_dim = int(emb1.shape[1])
cosine_similarity_raw = 1.0 - paired_cosine_distances(emb1, emb2)
predicted_score_0_5 = 2.5 * (cosine_similarity_raw + 1.0)
absolute_error = np.abs(predicted_score_0_5 - labels)

print({
    "embedding_dim": embedding_dim,
    "emb1_shape": emb1.shape,
    "emb2_shape": emb2.shape,
})


In [ ]:
pearson_raw = pearsonr(cosine_similarity_raw, labels).statistic
spearman_raw = spearmanr(cosine_similarity_raw, labels).statistic
pearson_rescaled = pearsonr(predicted_score_0_5, labels).statistic
spearman_rescaled = spearmanr(predicted_score_0_5, labels).statistic

results_df = bucket_df.copy()
results_df["cosine_similarity_raw"] = cosine_similarity_raw
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error

abs_error_summary = {
    "mean_absolute_error": float(np.mean(absolute_error)),
    "median_absolute_error": float(np.median(absolute_error)),
    "p90_absolute_error": float(np.percentile(absolute_error, 90)),
    "max_absolute_error": float(np.max(absolute_error)),
}

bucket_metadata = {
    "selected_bucket": selected_bucket,
    "bucket_size": int(len(bucket_df)),
    "sentence1_len_mean": float(bucket_df["sentence1_len"].mean()),
    "sentence2_len_mean": float(bucket_df["sentence2_len"].mean()),
    "max_pair_len_min": int(bucket_df["max_pair_len"].min()),
    "max_pair_len_max": int(bucket_df["max_pair_len"].max()),
}

print(bucket_metadata)
print(abs_error_summary)
print(results_df[["sentence1", "sentence2", "label", "cosine_similarity_raw", "predicted_score_0_5", "absolute_error"]].head(10))


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"bucket_column: {bucket_column}")
print(f"selected_bucket: {selected_bucket}")
print(f"full_validation_examples: {len(df)}")
print(f"bucket_examples: {len(bucket_df)}")
print(f"embedding_dimensionality: {embedding_dim}")
print(f"pearson_raw_cosine: {pearson_raw:.6f}")
print(f"spearman_raw_cosine: {spearman_raw:.6f}")
print(f"pearson_rescaled_0_5: {pearson_rescaled:.6f}")
print(f"spearman_rescaled_0_5: {spearman_rescaled:.6f}")
print(f"mean_absolute_error: {abs_error_summary['mean_absolute_error']:.6f}")
print(f"median_absolute_error: {abs_error_summary['median_absolute_error']:.6f}")
print(f"p90_absolute_error: {abs_error_summary['p90_absolute_error']:.6f}")
print(f"max_absolute_error: {abs_error_summary['max_absolute_error']:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
